# 3. Machine Learning Modeling (Medallion: Gold Layer)
In questo notebook completeremo la pipeline di data science (la "Gold" layer della Medallion Architecture) implementando e confrontando diversi modelli di classificazione.
L'obiettivo è prevedere l'esito di una partita (`outcome`) sulla base delle feature statistiche e geometriche estratte.

## I modelli analizzati
Come discusso nella fase di progettazione, metteremo a confronto:
1. **Random Forest Classifier**: Scelta principale. Gestisce bene dati tabulari eterogenei, riduce l'overfitting rispetto a un albero singolo (con sole 760 righe il DT overfitta facilmente) e fornisce le *Feature Importances* per interpretare i risultati.
2. **Support Vector Machine — Linear kernel**: Confronta la separabilità delle classi in spazio lineare.
3. **Support Vector Machine — RBF kernel**: Verifica se le interazioni tattiche non-lineari richiedono un confine di separazione più complesso.
4. **Decision Tree Classifier**: Modello di riferimento interpretabile (baseline albero singolo).
5. **Gaussian Naive Bayes (GNB)**: Modello probabilistico estremamente semplice.

> **💡 Nota importante per l'Esame — Perché testare entrambi i kernel SVM?**
> Il calcio è uno sport con interazioni tattiche complesse e non lineari (es. il valore di un passaggio dipende dalla zona, dal contesto, dal tempo di gioco). Abbiamo voluto verificare empiricamente se queste relazioni richiedano effettivamente un kernel non lineare (RBF), oppure se le feature tattiche aggregate siano sufficientemente "discriminanti in linea" da rendere il kernel lineare adeguato. Confrontare i due risultati dimostra rigore sperimentale.

> **💡 Nota importante per l'Esame — Il Paradosso del Gaussian Naive Bayes:**
> Dal punto di vista teorico, il GNB assume che tutte le feature siano *indipendenti* data la classe, un'assunzione chiaramente falsa nel calcio (es. il numero di passaggi è fortemente correlato al field tilt). Inoltre, assume distribuzioni gaussiane (spesso non vera per conteggi come falli o cartellini).
> **Tuttavia**, proprio in scenari ad altissima varianza (rumore) e con classi fortemente sovrapposte come nel calcio sportivo, i modelli più complessi tendono a fare overfitting sul rumore. In questi casi, il "bias" introdotto dalle assunzioni ingenue del Naive Bayes funge da forte regolarizzatore naturale, rendendo il GNB sorprendentemente robusto e, in alcuni casi, capace di superare in generalizzazione modelli ben più complessi!


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')


## 1. Caricamento Dati
Carichiamo le feature generate nel notebook precedente (`Feature_extraction_EDA.ipynb`).

In [ ]:
# Caricamento del dataset
df = pd.read_parquet('../data/processed/features.parquet')
print(f"Dimensioni del dataset: {df.shape}")
df.head()


## 2. Preprocessing e Prevenzione del Data Leakage
Per prevedere il risultato della partita (`outcome`) sulla base dello stile di gioco e delle statistiche, dobbiamo assicurarci di rimuovere tutte le variabili che "svelano" direttamente o indirettamente il risultato. Questo problema si chiama **Data Leakage**.

Variabili da rimuovere:
- `goals_scored` e `goals_conceded`: Se il modello conosce i gol, l'accuratezza sarà del 100% per ovvi motivi, rendendo inutile l'analisi tattica.
- `match_id` e `teamId`: Identificativi non predittivi o che potrebbero memorizzare i risultati storici del campionato (overfitting).


In [ ]:
# Colonne da eliminare per prevenire Data Leakage
cols_to_drop = ['match_id', 'teamId', 'goals_scored', 'goals_conceded']

X = df.drop(columns=cols_to_drop + ['outcome'])
y = df['outcome']

print("Feature utilizzate per la classificazione:")
print(X.columns.tolist())


### Suddivisione in Train/Test e Standardizzazione
- **Train/Test split**: 80% addestramento, 20% test.
- **Standardizzazione (Z-score normalization)**: Modelli come SVM e, in misura minore, GNB e Decision Trees, beneficiano di avere tutte le feature sulla stessa scala (media 0, varianza 1). Random Forest è invariante rispetto alla scala, ma scalarli non danneggia le sue performance.

In [ ]:
# Train/Test Split con stratificazione (mantiene la proporzione di Win/Loss/Draw)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Inizializziamo lo StandardScaler
scaler = StandardScaler()

# Fit solo sul train per evitare information leakage dal test set
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convertiamo di nuovo in dataframe per mantenere i nomi delle colonne (utile per la feature importance)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print(f"Train size: {X_train_scaled.shape[0]}")
print(f"Test size: {X_test_scaled.shape[0]}")


## 3. Addestramento e Valutazione dei Modelli
Andiamo ora ad istanziare i cinque modelli e confrontarli in modo sistematico.

> **⚠️ Nota sul preprocessing per SVM:** A differenza di Decision Tree e Random Forest (invarianti rispetto alla scala delle feature), la SVM **richiede obbligatoriamente** la standardizzazione. Le nostre feature hanno scale molto diverse: `n_passes` può arrivare a ~600, mentre `pass_success_rate` è compresa tra 0 e 1. Senza `StandardScaler`, le feature ad alta magnitudine dominerebbero il calcolo della distanza dal margine, rendendo il modello inutilizzabile. Per coerenza, applichiamo lo stesso scaler scalato su tutti i modelli.

> **⚠️ `class_weight='balanced'`:** Utilizziamo questo parametro su SVM e Decision Tree per compensare lo sbilanciamento tra le classi (Win e Loss sono più frequenti dei Draw nel dataset). In questo modo l'algoritmo pesa proporzionalmente gli errori sulle classi meno rappresentate, evitando di ignorare i pareggi.


In [ ]:
# Definizione dei modelli
# SVM è testata con due kernel per rigore sperimentale:
# - linear: verifica se la separazione lineare è sufficiente
# - rbf:    verifica se servono confini non lineari (più espressivo)
models = {
    "Random Forest":    RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    "SVM (linear)":     SVC(kernel='linear', class_weight='balanced', random_state=42),
    "SVM (RBF)":        SVC(kernel='rbf',    class_weight='balanced', random_state=42),
    "Decision Tree":    DecisionTreeClassifier(max_depth=10, class_weight='balanced', random_state=42),
    "Gaussian Naive Bayes": GaussianNB()
}

results = {}

# Funzione per plottare la confusion matrix
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=['Win', 'Draw', 'Loss'])
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Win', 'Draw', 'Loss'],
                yticklabels=['Win', 'Draw', 'Loss'])
    plt.title(f'Confusion Matrix — {title}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

# Addestramento, previsione e valutazione
for name, model in models.items():
    print(f"\n{'='*55}")
    print(f"  Training: {name}")
    print(f"{'='*55}")

    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print(f"Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Win', 'Draw', 'Loss']))

    plot_confusion_matrix(y_test, y_pred, name)


## 4. Confronto delle Performance
Visualizziamo le accuratezze ottenute per valutare in modo diretto quale modello ha generalizzato meglio.

In [ ]:
# Grafico a barre delle accuratezze
plt.figure(figsize=(10, 6))
sns.barplot(x=list(results.values()), y=list(results.keys()), palette='viridis')
plt.title('Confronto Accuratezza dei Modelli')
plt.xlabel('Accuracy')
plt.xlim(0, 1.0)
for index, value in enumerate(results.values()):
    plt.text(value + 0.01, index, f"{value:.4f}", va='center')
plt.show()


## 5. Analisi della Feature Importance (Random Forest)
Una delle ragioni principali per l'utilizzo del Random Forest in progetti di business/sports analytics è l'interpretabilità globale, ovvero la possibilità di estrarre quali feature hanno influenzato maggiormente le previsioni.

In [ ]:
# Estraiamo l'importanza dal modello Random Forest
rf_model = models["Random Forest"]
importances = rf_model.feature_importances_

# Creiamo un DataFrame per una migliore visualizzazione
feat_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Plottiamo le top 15 feature
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=feat_imp.head(15), palette='magma')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.xlabel('Relative Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()
